# Fine-tune Mamba to be Yo-bot 🐍

Fine-tunes a Mamba state-space model on Q&A pairs about Chaiyo.

**Cost: $0** — run on a free Colab T4 GPU (`Runtime → Change runtime type → T4 GPU`).

**For best results, first run `python3 augment.py` locally** (needs your `GROQ_API_KEY`) to multiply `dataset.jsonl` into `dataset_augmented.jsonl` (~300–500 pairs). More phrasings = a smarter, more natural tiny model.

Steps: install → pick model size → load dataset → train → test → push to Hugging Face Hub → deploy `hf-space/`.

In [ ]:
%pip install -q -U transformers datasets accelerate
# Optional speedup (CUDA kernels). Fine-tuning works without them (slower):
# %pip install -q mamba-ssm causal-conv1d

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Pick your size — all fine-tune free on a T4:
#   'state-spaces/mamba-130m-hf'  fast replies on free CPU Space, roughest quality
#   'state-spaces/mamba-370m-hf'  noticeably more fluent, ~2-3x slower serving  ← good default with augmented data
#   'state-spaces/mamba-790m-hf'  best fluency, slow on free CPU (only if patient)
MODEL_ID = 'state-spaces/mamba-370m-hf'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device)
print(f'{sum(p.numel() for p in model.parameters())/1e6:.0f}M params on {device}')

In [ ]:
# Upload dataset_augmented.jsonl (preferred, from augment.py) or dataset.jsonl.
import os
from google.colab import files
uploaded = files.upload()

DATA_FILE = 'dataset_augmented.jsonl' if os.path.exists('dataset_augmented.jsonl') else 'dataset.jsonl'

from datasets import load_dataset
ds = load_dataset('json', data_files=DATA_FILE, split='train').shuffle(seed=42)
print(f'{len(ds)} pairs from {DATA_FILE}')
print(ds[0]['text'])

In [ ]:
MAX_LEN = 128  # Q&A pairs are short; smaller = far less GPU memory

def tokenize(batch):
    # Append EOS so the model learns where answers END (critical for a tiny model).
    texts = [t + tokenizer.eos_token for t in batch['text']]
    out = tokenizer(texts, truncation=True, max_length=MAX_LEN, padding='max_length')
    out['labels'] = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in ids]
        for ids in out['input_ids']
    ]
    return out

tokenized = ds.map(tokenize, batched=True, remove_columns=ds.column_names)

In [ ]:
from transformers import Trainer, TrainingArguments

# With an augmented dataset (300+ pairs) fewer epochs are needed — the
# variety does the teaching, not repetition.
EPOCHS = 4 if len(ds) > 200 else 15

args = TrainingArguments(
    output_dir='yo-bot-mamba',
    num_train_epochs=EPOCHS,
    # Small batch + accumulation = same effective batch of 8, but fits
    # the free T4 (transformers' Mamba slow-path is memory-hungry
    # without the mamba-ssm CUDA kernels).
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy='no',
    fp16=torch.cuda.is_available(),
    report_to='none',
)

trainer = Trainer(model=model, args=args, train_dataset=tokenized)
trainer.train()

In [ ]:
# Humanized sampling — same settings the Space server uses. Sampling
# (vs greedy) is what makes replies feel alive instead of robotic.
def ask(question, max_new_tokens=90):
    prompt = f'Q: {question}\nA:'
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.3,
        eos_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    reply = text.split('A:', 1)[-1].strip()
    return reply.split('\nQ:', 1)[0].strip()

for q in ['hi', 'Who is Chaiyo?', 'Where does Chaiyo work?', 'What is Gig&Co?',
          'Are you ChatGPT?', 'ชัยโยเก่งอะไรบ้าง']:
    print('Q:', q)
    print('A:', ask(q))
    print()

In [ ]:
# Push to your Hugging Face account (free). Get a WRITE token at
# https://huggingface.co/settings/tokens
from huggingface_hub import notebook_login
notebook_login()

REPO = 'yovatcha/yo-bot-mamba'  # change to your username
model.push_to_hub(REPO)
tokenizer.push_to_hub(REPO)
print('Done → set MODEL_ID env var on your HF Space to', REPO)

## Tuning it your way
- **More human** → grow the dataset: more casual phrasings, more personality in answers, more small talk (`hi`, `thanks`, jokes). Re-run `augment.py` after editing `dataset.jsonl`.
- **Loops / rambles** → raise `repetition_penalty` (1.3 → 1.5), lower `max_new_tokens`, or train fewer epochs.
- **Too random / makes stuff up** → lower `temperature` (0.7 → 0.5) or switch `do_sample=False`.
- **Wrong facts** → add that exact Q&A (and paraphrases) to the dataset; tiny models only know what they've literally seen.
- **Overfitting check**: if every reply is a word-for-word dataset answer, drop epochs; if replies are gibberish, add epochs/data.
- Bigger ≠ always better on free CPU serving — 370m is the sweet spot; test reply latency on your Space before committing to 790m.